# Vanilla (Softmax) vs Linear Attention
같은 Q, K, V를 이용해 **출력, 정규화, 속도, 메모리**를 비교해 봅니다.

- Vanilla: `softmax(Q @ K.T / sqrt(D)) @ V`
- Linear: `phi(Q) @ (phi(K).T @ V) / (phi(Q) @ phi(K).sum(axis=0))`
- `phi(x) = ELU(x) + 1` (논문 2006.16236)
- 둘은 **서로 다른 유사도 함수**를 사용하므로 출력이 같지 않아도 정상입니다.
- 이 예제는 **non-causal**이며 실제 GPU 벤치마크가 아닙니다.

## 1. 공통 입력

In [ ]:
import numpy as np
from time import perf_counter
# 랜덤 난수 생성기 사용
rng = np.random.default_rng(7) 
# seq, kv dim, token_dim
N, D, M = 4, 3, 2
# 정규분포 뽑기
Q = rng.normal(size=(N, D))
K = rng.normal(size=(N, D))
V = rng.normal(size=(N, M))
# 특성변환 elu(x) + 1
def phi(x):
    return np.where(x > 0, x + 1, np.exp(x))
# shape 뽑기
print("Q",Q.shape,"K",K.shape,"V",V.shape)

Q (4, 3) K (4, 3) V (4, 2)


## 2. 두 Attention을 동일한 Q, K, V로 계산
Vanilla는 유사도마다 exp를 적용한 뒤 **행별 softmax**로 정규화합니다. Linear는 양의 feature map으로 유사도를 만들고 분배법칙으로 계산을 재배열합니다.

In [ ]:
# 기본 어텐션
def vanilla_attention(q, k, v):
    # qk 내적 이후, 길이의 제곱근로 나누어 너무 정답이 편향되는 것을 막기
    scores = (q @ k.T) / np.sqrt(q.shape[1])
    # 각 query에 대한 k 최댓값을 빼서 exp() 값이 1 이하로 만들며 각 행의 가중치(exp점수) 합을 1로 만들기
    weights = np.exp(scores - scores.max(axis=1, keepdims=True))
    # 각 query에 대해서 총합을 각 q-k 에 나누어 각 q열(axis=1)의 총 합이 1이 되게 만들기
    weights /= weights.sum(axis=1, keepdims=True)
    # 마지막 v와의 내적을 통해 반환해주며 exp() 가중치 행렬을 반환해줌
    return weights @ v, weights

# linear 어텐션
def linear_attention(q, k, v):
    # 계산하기 전에 elu 특성변환을 해주기
    qf, kf = phi(q), phi(k)
    # q @ k.T @ v 에서 먼저 각 q_i가 곱할 행렬을 만들어주기
    # kf(n, d: qk-dim) / v(n, m=seq-dim) 을 미리 행렬곱해주면
    # (d, m) 형태로 q_i(1, d) 가 내적을 통해 (1, m) 형태의 토큰을 바로 출력할 수 있음.
    S = kf.T @ v
    # q_i가 내적을 하더라도 동일하게 정규화를 해야함.
    # 따라서 (n, d) -> (d,) 형태로 q가 질의한 모든 토큰들의 집중도 합이 1이 되게 하기 위해 
    # 모든 토큰의 값을 미리 합쳐둠
    # 여기에서 z는 어짜피 모든 토큰의 벡터의 요소 위치별 합이기 때문에 새로운 토큰이 만들어지면
    # W_k에 한번 투영한 뒤, 누적해주면 됨.
    # S의 경우에도 동일하게 새로운 토큰에 대한 k값을 새로운 v와 내적해두어 누적하면 됨.
    # 기존 S (d, n) @ (n, m) 에서 추가되는 토큰 1개의 행렬은 (d, 1) @ (1, m) 이기 때문에 
    # 차원이 맞아서 설정 가능
    z = kf.sum(axis=0) # z는 (d)
    # 들어온 토큰들에 대해서 기존 누적 질의 / 들어온 토큰들 각가이 다른 토큰을 얼마나 보는지 행렬로 표현
    # 1. qf @ S = (n, d) @ (d, m) = (n, m)
    # 2. qf @ z = (n, d) @ (d, -) = (n, 1(: m))
    # 최종적으로 나누어주어 각 sum(axis=1) = 1 이 될 수 있게 제작
    return (qf @ S) / (qf @ z)[:, None]

# 바닐라 계산 토큰(들), 거기에서 모든 값들의 exp() 행렬
out_vanilla, softmax_weights = vanilla_attention(Q, K, V)
# 어텐션을 통해 n개 토큰에 대한 m차원 토큰 벡터를 만들기
out_linear = linear_attention(Q, K, V)
# 정규화되지 않은 커널
kernel_A = phi(Q) @ phi(K).T
# 정규화 시키기
kernel_weights = kernel_A / kernel_A.sum(axis=1, keepdims=True)
print("softmax weight (행별 합=1):\n", softmax_weights.round(3))
print("linear kernel weight (행별 합=1):\n", kernel_weights.round(3))
print("vanilla output:\n", out_vanilla.round(3))
print("linear output:\n", out_linear.round(3))
print("평균 출력 절대차:", np.abs(out_vanilla - out_linear).mean().round(5))

softmax weight (행별 합=1):
 [[0.241 0.24  0.302 0.217]
 [0.084 0.089 0.734 0.092]
 [0.269 0.225 0.318 0.188]
 [0.248 0.164 0.317 0.271]]
linear kernel weight (행별 합=1):
 [[0.309 0.33  0.082 0.278]
 [0.31  0.32  0.086 0.284]
 [0.311 0.323 0.104 0.263]
 [0.315 0.272 0.089 0.324]]
vanilla output:
 [[-0.914 -0.244]
 [-0.387 -0.025]
 [-0.827 -0.225]
 [-0.803 -0.228]]
linear output:
 [[-1.212 -0.359]
 [-1.195 -0.356]
 [-1.171 -0.346]
 [-1.135 -0.35 ]]
평균 출력 절대차: 0.30877


## 3. Linear는 직접 커널 Attention과 정확히 같은가?
**Vanilla와 Linear의 차이**는 유사도 함수 차이이고, 아래 두 식은 같은 커널 Attention의 계산 순서만 다릅니다.

In [3]:
direct_kernel = (kernel_A @ V) / kernel_A.sum(axis=1, keepdims=True)
print("직접 커널 방식 = Linear:", np.allclose(direct_kernel, out_linear))
assert np.allclose(direct_kernel, out_linear)

직접 커널 방식 = Linear: True


## 4. 토큰 길이에 따른 시간과 중간 행렬 메모리
NumPy 실행 시간은 CPU/BLAS 환경의 영향을 받고, 특히 작은 N에서는 Linear가 느릴 수도 있습니다. 중간 NxN score 행렬의 **이론상 크기**를 함께 봅니다. 여기서 softmax 방식의 NxN score+weight 메모리는 일시적으로 더 커질 수 있습니다.

In [20]:
# 함수를 repeat만큼 돌려서 성능을 확인합니다.
def median_ms(fn, q, k, v, repeats=3):
    fn(q, k, v)  # warmup
    times = []
    for _ in range(repeats):
        # 시작
        t0 = perf_counter()
        # 함수 싫행 (vanilla, linear)
        fn(q, k, v)
        # 종료 (밀리초 단위로 변환)
        times.append((perf_counter()-t0)*1000)
      
    return float(np.median(times))

print(f"{'N':>6} {'vanilla(ms)':>14} {'linear(ms)':>14} {'NxN(float64 MiB)':>18} {'S+z(MiB)':>12}")
for n in [64, 256, 1024]:
    # qkv 차원 모두 64로 설정
    d, m = 64, 64
    q, k, v = rng.normal(size=(n,d)), rng.normal(size=(n,d)), rng.normal(size=(n,m))
    # 어텐션 모두 실행해보기
    tv = median_ms(vanilla_attention,q,k,v)
    tl = median_ms(linear_attention,q,k,v)
    # attention score 한 행렬의 크기 -> (2**10)**2  -> MiB로 변환
    score_mem = n*n*8/(1024**2)
    # linear attention에서 S는 (d, m), z는 (d) 크기만큼 저장하기 때문에 이를 모두 계산
    state_mem = (d*m+d)*8/(1024**2)
    print(f"{n:6d} {tv:14.2f} {tl:14.2f} {score_mem:18.3f} {state_mem:12.3f}")

     N    vanilla(ms)     linear(ms)   NxN(float64 MiB)     S+z(MiB)
    64           0.09           0.15              0.031        0.032
   256           1.24           0.47              0.500        0.032
  1024          21.62           2.09              8.000        0.032


## 5. 결론
- **출력 품질:** 랜덤 QKV만으로 우열을 판단할 수 없습니다. 정확도 비교는 두 모델을 같은 작업·데이터·학습 예산으로 각각 학습하고 검증해야 합니다.
- **연산량:** Vanilla는 보통 O(N²D+N²M), Linear는 O(NDM+ND) (D,M 고정 시 N에 선형).
- **추론 상태:** Causal Linear Attention에서는 이전 토큰의 K,V를 전부 보관하지 않고 `S_t`, `z_t`를 누적할 수 있습니다.
- 이 노트북의 실행시간은 **비인과적 전체 시퀀스** 기준이며 실제 autoregressive decoding 속도와는 다릅니다.

In [18]:
perf_counter()

350006.370658